# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s for reference during extraction and analysis.

All record set information, field names, and columns reference their Croissant `@id` for clarity and reproducibility.

Let's list the record sets and their available fields:

In [ ]:
# List all available RecordSets and a summary of their Fields and Columns
if not metadata.record_sets:
    print("No RecordSets in this dataset; please check the dataset's schema.")
else:
    for rs in metadata.record_sets:
        print(f'RecordSet: @id={rs.id} name={rs.name}')
        if rs.fields:
            for f in rs.fields:
                print(f"  Field: @id={f.id} name={f.name} type={f.data_type}")
                if hasattr(f, 'columns') and f.columns:
                    for c in f.columns:
                        print(f"    Column: @id={c.id} name={c.name}")
        else:
            print("  No fields found for this RecordSet.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the RecordSet and Field `@id`s from the overview above.

**Note:** Replace the values below with the actual `@id` of the desired RecordSet (as found above).

In [ ]:
# Example: Extract all records from available RecordSets
record_sets = [rs.id for rs in getattr(metadata, 'record_sets', [])]  # List all RecordSet @id
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns in RecordSet @{record_set_id}: ", df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps. This example filters records for a selected numeric field, normalizes it, and optionally groups by another field.

> Use the `@id` of the numeric field and (optionally) a categorical/grouping field from the Data Overview above.

In [ ]:
# If there are no record sets, skip; otherwise, select the first one for demonstration
if not dataframes:
    print("No dataframes loaded.")
else:
    # Choose a RecordSet and a numeric field for demonstration (update as needed)
    record_set_id = record_sets[0]
    df = dataframes[record_set_id]
    print(f"Data from RecordSet '@id': {record_set_id}")

    # Display columns to assist with manual selection
    print("Available columns:")
    print(df.columns.tolist())

    # Select a numeric field that exists (update the field id if needed)
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

    if numeric_field:
        print(f"Analyzing numeric field: {numeric_field}")
        threshold = df[numeric_field].quantile(0.5)
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records ({numeric_field} > {threshold}):")
        display(filtered_df.head())

        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Optionally, group by another field (first non-numeric field)
        group_field = None
        for col in df.columns:
            if not pd.api.types.is_numeric_dtype(df[col]) and col != numeric_field:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean of {numeric_field}):")
            display(grouped_df.head())
    else:
        print("No numeric field found in the selected RecordSet.")

## 5. Visualization
Visualize data distributions or relationships. Below is an example histogram and scatter plot of two numeric fields, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df = dataframes[record_sets[0]]
    numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if numeric_cols:
        plt.figure(figsize=(7, 4))
        sns.histplot(df[numeric_cols[0]].dropna(), bins=30, kde=True)
        plt.title(f'Histogram of {numeric_cols[0]} (@id)')
        plt.show()
        
        if len(numeric_cols) > 1:
            plt.figure(figsize=(6,6))
            sns.scatterplot(x=df[numeric_cols[0]], y=df[numeric_cols[1]])
            plt.title(f"Scatter plot: {numeric_cols[0]} vs {numeric_cols[1]}")
            plt.xlabel(numeric_cols[0])
            plt.ylabel(numeric_cols[1])
            plt.show()
    else:
        print("No numeric fields to visualize in the selected RecordSet.")
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated loading, exploring, and visualizing a Croissant-described dataset using the `mlcroissant` library.

- **Metadata and overview**: The dataset provides rich metadata and (if present) multiple record sets with their field definitions, all referenced by stable `@id` strings.
- **Data analysis**: We filtered and normalized a selected numeric field, and performed basic grouping and visualization. For deeper analysis, refer to specific field annotations and data documentation.

**Next steps**: Further feature engineering, modeling, or domain-specific analyses may be performed depending on the research question and the specific entities identified by their `@id`s.